# 02. Эксперименты на локальной валидации

Все числа ниже считаются на одном и том же сплите (`artifacts/val_split.pkl`, строится `cg.validation.make_split`):
8 000 запросов с уникальными текстами, 37,5 % из них встречаются в обучающей части A, корпус = корпус бенчмарка + релевантные
объявления валидации (196 518 объявлений). Метрика та же, что в задаче: Recall@50, усреднённый по запросам.
Кандидаты источников закэшированы скриптами из `scripts/`, здесь они только объединяются и оцениваются.

In [1]:
import sys; sys.path.insert(0, "../src")
import pickle, numpy as np, pandas as pd
from cg.config import ART
from cg.metrics import report
from cg.fusion import fuse
from cg.sources.memory import ClickMemory
from cg.boost import Booster, location_centroids
d = pickle.load(open(ART / "val_split.pkl", "rb"))
A, V, corpus, rel, seen_q = d["A"], d["V"], d["corpus"], d["rel"], d["seen_q"]
c1 = pickle.load(open(ART / "val_cands_v1.pkl", "rb"))        # память, добор по микрокатегориям, BM25 top-300
bm5k = pickle.load(open(ART / "val_cands_bm5000.pkl", "rb"))  # BM25 top-5000
dense = pickle.load(open(ART / "val_cands_dense.pkl", "rb"))  # e5-small top-1000
cidx = {i: n for n, i in enumerate(corpus.item_id)}
mem = ClickMemory(A, cidx); cent = location_centroids(corpus, A)
print(len(A), len(V), len(corpus))

489673 8000 196518


## 1. Отдельные источники

Память кликов почти бесполезна (мало пересечений с корпусом), BM25 и dense по отдельности дают около 0,4.

In [2]:
print("память кликов:      ", report(c1["mem"], rel, seen_q))
print("добор по микрокат.: ", report(c1["prior"], rel, seen_q))
print("BM25 top-300:       ", report(c1["bm25"].tolist(), rel, seen_q))
print("e5-small top-1000:  ", report(dense["dense_1000"].tolist(), rel, seen_q, ks=(50, 300, 1000)))

память кликов:       {'r@50': 0.0239, 'r@100': 0.0239, 'r@300': 0.0239, 'r@50_seen': 0.0607, 'r@50_unseen': 0.0018, 'seen_share': 0.375}
добор по микрокат.:  {'r@50': 0.055, 'r@100': 0.082, 'r@300': 0.1487, 'r@50_seen': 0.1403, 'r@50_unseen': 0.0038, 'seen_share': 0.375}


BM25 top-300:        {'r@50': 0.397, 'r@100': 0.4811, 'r@300': 0.6345, 'r@50_seen': 0.397, 'r@50_unseen': 0.397, 'seen_share': 0.375}


e5-small top-1000:   {'r@50': 0.3603, 'r@300': 0.6118, 'r@1000': 0.775, 'r@50_seen': 0.354, 'r@50_unseen': 0.364, 'seen_share': 0.375}


## 2. Бусты по локации, расстоянию и популярности

Скор кандидата умножается на `(1 + b*[та же локация]) * (1 + d*exp(-dist/scale)) * (1 + c*log(1 + клики в train))`.
Локация даёт главный прирост: с 0,40 до 0,60 при том же пуле из 300 кандидатов.

In [3]:
def run(pool_k, **bk):
    bo = Booster(corpus, mem.item_pop, cent, **bk); bo.prepare(V)
    return report(fuse({"bm25": bm5k["bm25_5000"][:, :pool_k].tolist()}, {"bm25": 1.0}, forced=c1["mem"], quota=5, filler=c1["prior"], boost=bo), rel, seen_q)
print("пул 300, без бустов:          ", run(300, b=0, c=0, d=0))
print("пул 300, локация:             ", run(300, b=5, c=0, d=0))
print("пул 300, локация+расст.+попул:", run(300, b=5, c=0.5, d=5, scale_km=100))

пул 300, без бустов:           {'r@50': 0.4042, 'r@100': 0.4042, 'r@300': 0.4042, 'r@50_seen': 0.4167, 'r@50_unseen': 0.3968, 'seen_share': 0.375}


пул 300, локация:              {'r@50': 0.5854, 'r@100': 0.5854, 'r@300': 0.5854, 'r@50_seen': 0.612, 'r@50_unseen': 0.5694, 'seen_share': 0.375}


пул 300, локация+расст.+попул: {'r@50': 0.6016, 'r@100': 0.6016, 'r@300': 0.6016, 'r@50_seen': 0.6303, 'r@50_unseen': 0.5844, 'seen_share': 0.375}


## 3. Размер пула кандидатов

С бустами качество упирается в полноту пула BM25, поэтому пул расширяется, а в финале скорится весь корпус.

In [4]:
for k in (300, 1000, 2000, 5000):
    print(f"пул {k}: ", run(k, b=5, c=0.5, d=5, scale_km=100)["r@50"])

пул 300:  0.6016


пул 1000:  0.7073


пул 2000:  0.7451


пул 5000:  0.7618


## 4. Полный скоринг корпуса, BM25 + dense, реранкер

Дальше кандидаты не ограничиваются пулом: для каждого запроса считаются BM25 и косинус e5 по всем объявлениям
(`cg.scoring.FullScorer`), скоры нормируются в (0, 1] и складываются, затем применяются бусты.
Результаты на случайной подвыборке из 2 500 запросов валидации (`scripts/val_tune.py`, `scripts/val_full.py`):

| Вариант | Recall@50 |
|---|---|
| BM25 по всему корпусу + бусты | 0,79 |
| e5-small по всему корпусу + бусты | 0,78 |
| BM25 + e5, tau = 10 / 20 / 50 / 100 / 200 | 0,79 / 0,82 / **0,82** / 0,81 / 0,79 |
| RRF по рангам вместо суммы скоров | 0,82 |

Топ-300 полного скоринга (Recall@300 = 0,92) переупорядочивается LightGBM-классификатором на 23 признаках
(`cg.rerank.FEATS`): 2 фолда по запросам, `scripts/rerank_train.py`.

In [5]:
pool = pickle.load(open(ART / "val_rerank_pool.pkl", "rb"))
print("пул полного скоринга:", report(pool["top"], rel, seen_q, ks=(50, 100, 300)))
print(open(ART / "rerank_train.log").read().split("pool r@50/300")[1][:1200])

пул полного скоринга: {'r@50': 0.8235, 'r@100': 0.8698, 'r@300': 0.9194, 'r@50_seen': 0.8417, 'r@50_unseen': 0.8126, 'seen_share': 0.375}
: {'r@50': 0.8235, 'r@100': 0.8698, 'r@300': 0.9194, 'r@50_seen': 0.8417, 'r@50_unseen': 0.8126, 'seen_share': 0.375}
features (2400000, 23) pos 7355 25
fold 0: rerank {'r@50': 0.851, 'r@100': 0.851, 'r@300': 0.851, 'r@50_seen': 0.8645, 'r@50_unseen': 0.8429, 'seen_share': 0.374} | base 0.8253 51
importance [('dense_exp', np.int32(3306)), ('reviews', np.int32(3145)), ('mc_prob', np.int32(3096)), ('dense_cos', np.int32(3074)), ('bm25_norm', np.int32(2869)), ('log_price', np.int32(2683)), ('title_len', np.int32(2562)), ('rank', np.int32(2472)), ('dist_km', np.int32(2412)), ('rating', np.int32(2301))]
fold 1: rerank {'r@50': 0.841, 'r@100': 0.841, 'r@300': 0.841, 'r@50_seen': 0.8569, 'r@50_unseen': 0.8315, 'seen_share': 0.376} | base 0.8217 81
importance [('reviews', np.int32(3311)), ('dense_exp', np.int32(3228)), ('mc_prob', np.int32(3107)), ('dense_co